# FaceGuard AI — face mask detection (mask / no mask)

**Owner:** Nam Khánh (individual feature) · **Task:** mask · **Model:** MobileNetV3-Small vs MobileNetV3-Large vs MobileNetV4-Conv-Medium

Runs on **Google Colab** (Runtime → Change runtime type → **T4 GPU**) or locally. Code lives in `src/mask/`
(see `src/mask/README.md`); this notebook only calls it, so results match the command line.

**Data:** FMLD labels (Batagelj et al., 2021) on MAFA (Ge et al., 2017) and WIDER FACE (Yang et al., 2016) photos.

**References:** FMLD github.com/borutb-fri/FMLD · MobileNetV3 (Howard et al., 2019) · MobileNetV4 (Qin et al., 2024), weights from `timm`

## 1. Setup
Clones the repo, installs requirements and mounts Drive. If `src/mask` is not on the branch yet,
upload `mask_code.zip` to `MyDrive/FaceGuard/` and it is unpacked into the repo.

In [ ]:
import os, sys, subprocess, zipfile, importlib.util
from pathlib import Path

REPO_URL = "https://github.com/henryhoang207/Facial-Recognition---COS30082.git"
BRANCH = "main"  # đổi thành branch có src/mask nếu đã push

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/repo")
    DRIVE_DIR = Path("/content/drive/MyDrive/FaceGuard")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    # Colab có sẵn torch, numpy, pandas, sklearn...; chỉ cài gói nào còn thiếu
    needed = {"cv2": "opencv-python-headless", "sklearn": "scikit-learn", "tqdm": "tqdm",
              "pandas": "pandas", "matplotlib": "matplotlib", "PIL": "Pillow", "timm": "timm"}
    missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DRIVE_DIR = REPO_DIR

if not (REPO_DIR / "src" / "mask" / "train.py").exists():
    code_zip = DRIVE_DIR / "mask_code.zip"
    assert code_zip.exists(), f"src/mask missing: push it to {BRANCH} or upload {code_zip}"
    zipfile.ZipFile(code_zip).extractall(REPO_DIR)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
# dữ liệu để trên ổ của Colab (nhanh hơn Drive nhiều); checkpoint + hình lưu về Drive
DATA_DIR = Path("/content/data") if IN_COLAB else REPO_DIR / "data"
CKPT_DIR = DRIVE_DIR / "checkpoints" if IN_COLAB else REPO_DIR / "model" / "Mask"
FIG_DIR = DRIVE_DIR / "figures" if IN_COLAB else REPO_DIR / "reports" / "figures"
for d in (DATA_DIR, CKPT_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("repo:", REPO_DIR, "| data:", DATA_DIR, "| checkpoints:", CKPT_DIR)

In [ ]:
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE,
      torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "(no GPU: Runtime -> Change runtime type -> T4 GPU)")

## 2. Config
`DATA_SOURCE = "download"`: Colab downloads FMLD (GitHub), WIDER FACE (Hugging Face) and MAFA (Kaggle) itself, then crops faces here.
MAFA needs a Kaggle API token: put `kaggle.json` in `MyDrive/FaceGuard/`, or add Colab secrets `KAGGLE_USERNAME` and `KAGGLE_KEY`.

`DATA_SOURCE = "zip"`: run `python -m src.mask.prepare` on your laptop, zip `data/mask_faces` to
`mask_faces.zip`, upload it to `MyDrive/FaceGuard/`. Skips all downloads and the crop step.

In [ ]:
DATA_SOURCE = "download"   # "download" | "zip"
CONFIG = {
    "task": "mask",
    "img_size": 224,          # cố định trong src/mask (crop_face scale 1.3)
    "epochs": 8,
    "batch_small": 128,       # T4 16 GB; dùng 64 trên GPU 4 GB
    "batch_large": 96,
    "batch_v4m": 96,
    "workers": 2,             # Colab có 2 vCPU
}
FACES_DIR = DATA_DIR / "mask_faces"
CONFIG

## 3. Data
Download (or unzip), then crop every FMLD face with the shared `crop_face` (224 px, scale 1.3).
Skipped automatically if `mask_faces/manifest.csv` already exists.

In [ ]:
import shutil

def sh(cmd: str):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

def kaggle_auth():
    token = DRIVE_DIR / "kaggle.json"
    if token.exists():
        target = Path.home() / ".kaggle" / "kaggle.json"
        target.parent.mkdir(exist_ok=True)
        shutil.copy(token, target)
        target.chmod(0o600)
        return
    if IN_COLAB:
        from google.colab import userdata
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

if (FACES_DIR / "manifest.csv").exists():
    print("faces already prepared")
elif DATA_SOURCE == "zip":
    zipfile.ZipFile(DRIVE_DIR / "mask_faces.zip").extractall(DATA_DIR)
else:
    if not (DATA_DIR / "FMLD_annotations").exists():
        sh(f"git clone -q --depth 1 https://github.com/borutb-fri/FMLD.git /content/FMLD")
        sh(f"unzip -q -o /content/FMLD/FMLD_annotations.zip -d {DATA_DIR}")
    wider = DATA_DIR / "WIDER"
    for part in ("WIDER_train", "WIDER_val"):
        if not (wider / part).exists():
            url = f"https://huggingface.co/datasets/CUHK-CSE/wider_face/resolve/main/data/{part}.zip"
            sh(f"wget -q -O /content/{part}.zip '{url}' && unzip -q /content/{part}.zip -d {wider} && rm /content/{part}.zip")
    if not (DATA_DIR / "MAFA").exists():
        kaggle_auth()
        sh("pip install -q kaggle")
        sh(f"kaggle datasets download -d rahulmangalampalli/mafa-data -p /content --unzip -o -q")
        # zip Kaggle chứa train-images/, test-images/, MAFA-Label-* ở gốc
        (DATA_DIR / "MAFA").mkdir()
        for name in ("train-images", "test-images"):
            shutil.move(f"/content/{name}", DATA_DIR / "MAFA" / name)
    sh(f"python -m src.mask.prepare --data-root {DATA_DIR} --out {FACES_DIR} --workers {os.cpu_count()}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

manifest = pd.read_csv(FACES_DIR / "manifest.csv")
display(manifest.groupby(["split", "label"]).size().unstack())

fig, axes = plt.subplots(2, 8, figsize=(16, 4.4))
for row, label in enumerate(["mask", "no_mask"]):
    sample = manifest[(manifest.split == "train") & (manifest.label == label)].sample(8, random_state=1)
    for ax, path in zip(axes[row], sample.path):
        ax.imshow(Image.open(FACES_DIR / path)); ax.set_axis_off()
    axes[row][0].set_title(label, loc="left")
plt.tight_layout(); plt.show()

## 4. Model
ImageNet **MobileNetV3-Small** (fast, ~2.5 M parameters), **MobileNetV3-Large**, and the newer
**MobileNetV4-Conv-Medium** (2024, ~8.4 M parameters, `timm` weights `mobilenetv4_conv_medium.e500_r224_in1k`).
The 1000-class head is replaced by a 2-class layer and all layers are fine-tuned: masks are a simple, local cue,
so a small pretrained network should be enough, and Small keeps the app above 10 FPS on a laptop CPU.
MobileNetV4 checks whether a newer, larger backbone buys accuracy worth its extra CPU time.

In [ ]:
from src.mask.model import build_model
for arch in ("mobilenetv3s", "mobilenetv3l", "mobilenetv4m"):
    m = build_model(arch, pretrained=False)
    print(arch, f"{sum(p.numel() for p in m.parameters()) / 1e6:.2f} M parameters")

## 5. Train
Each cell trains one model; run only the ones you need (checkpoints from earlier runs stay on Drive).
AdamW, warmup + cosine LR, label smoothing 0.05, mixed precision, early stop on val macro F1 (patience 3).

In [ ]:
sh(f"python -m src.mask.train --arch mobilenetv3s --faces {FACES_DIR} --epochs {CONFIG['epochs']} "
   f"--batch-size {CONFIG['batch_small']} --workers {CONFIG['workers']} --ckpt-dir {CKPT_DIR} --figures {FIG_DIR}")

In [ ]:
sh(f"python -m src.mask.train --arch mobilenetv3l --faces {FACES_DIR} --epochs {CONFIG['epochs']} "
   f"--batch-size {CONFIG['batch_large']} --workers {CONFIG['workers']} --ckpt-dir {CKPT_DIR} --figures {FIG_DIR}")

In [ ]:
sh(f"python -m src.mask.train --arch mobilenetv4m --faces {FACES_DIR} --epochs {CONFIG['epochs']} "
   f"--batch-size {CONFIG['batch_v4m']} --workers {CONFIG['workers']} --ckpt-dir {CKPT_DIR} --figures {FIG_DIR}")

## 6. Evaluate
FMLD test split only: accuracy, macro F1, AUC, recall per class, per source (MAFA / WIDER), CPU ms per face.

In [ ]:
from datetime import date
from IPython.display import Image as Show, Markdown

ARCHS = ("mobilenetv3s", "mobilenetv3l", "mobilenetv4m")
for arch in ARCHS:
    found = sorted(CKPT_DIR.glob(f"mask_{arch}_*.pt"))
    if not found:
        print("no checkpoint for", arch)
        continue
    ckpt = found[-1]
    sh(f"python -m src.mask.evaluate --checkpoint {ckpt} --faces {FACES_DIR} --figures {FIG_DIR} --workers {CONFIG['workers']}")

display(Markdown((FIG_DIR / "mask_results.md").read_text()))
for arch in ARCHS:
    for name, width in (("training", 700), ("confusion", 320)):
        figure = FIG_DIR / f"mask_{arch}_{name}.png"
        if figure.exists():
            display(Show(str(figure), width=width))
        else:
            print("missing", figure.name)

## 7. Results and notes
Copy the final checkpoint from `MyDrive/FaceGuard/checkpoints/` to `model/Mask/` on your laptop for the app,
and the figures from `MyDrive/FaceGuard/figures/` to `reports/figures/`.

| Model | Accuracy | Macro F1 | AUC | CPU ms / face | Notes |
| --- | --- | --- | --- | --- | --- |
| MobileNetV3-Small |  |  |  |  |  |
| MobileNetV3-Large |  |  |  |  |  |
| MobileNetV4-Conv-Medium |  |  |  |  |  |

Notes to fill in: FMLD counts scarves and pulled-up collars as `masked_face`; check MAFA vs WIDER accuracy (source bias);
add webcam photos of the team with `--extra` for the demo condition.